In [3]:
import os
from os.path import join
import trimesh
from itertools import combinations
import numpy as np
import open3d as o3d
import torch
from tqdm import tqdm
import logging
logging.getLogger("trimesh").setLevel(logging.ERROR)

def to_o3d_pcd(pts):
    '''
    From numpy array, make point cloud in open3d format
    :param pts: point cloud (nx3) in numpy array
    :return: pcd: point cloud in open3d format
    '''
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd
    
def get_correspondences(src_pcd, tgt_pcd, search_voxel_size, K=None):
    '''
    Give source & target point clouds as well as the relative transformation between them, calculate correspondences according to give threshold
    :param src_pcd: source point cloud
    :param tgt_pcd: target point cloud
    :param search_voxel_size: given threshold
    :param K: if k is not none, select top k nearest neighbors from candidate set after radius search
    :return: (m, 2) torch tensor, consisting of m correspondences
    '''

    pcd_tree = o3d.geometry.KDTreeFlann(tgt_pcd)

    correspondences = []
    for i, point in enumerate(src_pcd.points):
        [count, idx, _] = pcd_tree.search_radius_vector_3d(point, search_voxel_size)
        if K is not None:
            idx = idx[:K]
        for j in idx:
            correspondences.append([i, j])

    correspondences = np.array(correspondences)
    correspondences = torch.from_numpy(correspondences)
    return correspondences
    
datapath = '../../../data/bbad_v2'
data_category = 'everyday'
split = 'val'
filepaths = join('../data/data_list', f"{data_category}_{split}.txt")

with open(filepaths, 'r') as f:
    filepaths = [x.strip() for x in f.readlines() if x.strip()]

filepaths = [x for x in filepaths if 2 <= int(x.split()[0]) <= 20]
n_frac_list = [int(x.split()[0]) for x in filepaths]
filepaths_list = [x.split()[1] for x in filepaths]

In [4]:
f = open(f"./mpa_{data_category}_{split}.txt", 'w')
f.close()

for idx in tqdm(range(len(filepaths_list))):
    filepath = filepaths_list[idx]
    n_frac = n_frac_list[idx]
    
    base_path = join(datapath, filepath)
    obj_paths = [join(base_path, x) for x in os.listdir(base_path)]
    meshes = [trimesh.load_mesh(x) for x in obj_paths]
    mesh_areas = [mesh_.area for mesh_ in meshes]
    total_area = sum(mesh_areas)
    pcds = []
    for mesh in meshes:
        n_pts = int(5000 * mesh.area / total_area)
        sampled_pts = torch.tensor(trimesh.sample.sample_surface_even(mesh, n_pts)[0]).float()
    
        if sampled_pts.size(0) < 256:
            extra_pts, _ = trimesh.sample.sample_surface(mesh, 256 - sampled_pts.size(0))
            sampled_pts = torch.cat([sampled_pts, torch.tensor(extra_pts).float()], dim=0)
                
        pcds.append(sampled_pts)
    
    elements = list(range(0, n_frac))
    combinations_list = list(combinations(elements, 2))
    
    
    for comb in combinations_list:
        src_pcd, trg_pcd = pcds[comb[0]], pcds[comb[1]]
        num_corr = get_correspondences(to_o3d_pcd(src_pcd), to_o3d_pcd(trg_pcd), 0.018).size(0)
        if num_corr > 128:
            with open(f"./mpa_{data_category}_{split}.txt", 'a') as f:
                data = f"{n_frac:03} {'/'.join(base_path.split('/')[5:])} {obj_paths[comb[0]].split('/')[-1]} {obj_paths[comb[1]].split('/')[-1]} \n"
                f.write(data)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7872/7872 [32:54<00:00,  3.99it/s]
